In [ ]:

from pyspark.sql import functions as F

# Read the original 2025 flight records.
# This notebook performs read-only analysis.

raw_path = (
    "abfss://raw@stflight60106354.dfs.core.windows.net/"
    "flight_delays/2025/csv/"
)

raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "FAILFAST")
    .csv(raw_path)
)

df = (
    raw
    .withColumn("YearNum", F.col("Year").cast("int"))
    .withColumn("MonthNum", F.col("Month").cast("int"))
    .withColumn("FlightDateParsed", F.to_date("FlightDate"))
    .withColumn("CancelledNum", F.col("Cancelled").cast("double"))
    .withColumn("DivertedNum", F.col("Diverted").cast("double"))
    .withColumn("ArrDelayNum", F.col("ArrDelay").cast("double"))
    .withColumn("TargetNum", F.col("ArrDel15").cast("double"))
    .withColumn("CRSDepNum", F.col("CRSDepTime").cast("int"))
)

# Reproduce the production ETL validation rules.

valid_date = (
    (F.col("YearNum") == 2025)
    & F.col("MonthNum").between(1, 12)
    & F.col("FlightDateParsed").isNotNull()
    & (F.year("FlightDateParsed") == F.col("YearNum"))
    & (F.month("FlightDateParsed") == F.col("MonthNum"))
    & (
        F.dayofmonth("FlightDateParsed")
        == F.col("DayofMonth").cast("int")
    )
)

valid_airports = (
    F.col("Reporting_Airline").isNotNull()
    & (F.trim("Reporting_Airline") != "")
    & F.col("Origin").isNotNull()
    & (F.trim("Origin") != "")
    & F.col("Dest").isNotNull()
    & (F.trim("Dest") != "")
)

valid_time = (
    (
        F.col("CRSDepNum").between(0, 2359)
        & ((F.col("CRSDepNum") % 100) < 60)
    )
    | (F.col("CRSDepNum") == 2400)
)

valid_status = (
    (F.col("CancelledNum") == 0)
    & (F.col("DivertedNum") == 0)
)

valid_arrival = F.col("ArrDelayNum").isNotNull()

valid_target = (
    F.col("TargetNum").isin(0.0, 1.0)
    & (
        F.col("TargetNum")
        == F.when(
            F.col("ArrDelayNum") >= 15, 1.0
        ).otherwise(0.0)
    )
)

df = (
    df
    .withColumn(
        "Reason_InvalidDate",
        ~F.coalesce(valid_date, F.lit(False))
    )
    .withColumn(
        "Reason_MissingAirlineOrAirport",
        ~F.coalesce(valid_airports, F.lit(False))
    )
    .withColumn(
        "Reason_InvalidDepartureTime",
        ~F.coalesce(valid_time, F.lit(False))
    )
    .withColumn(
        "Reason_Cancelled",
        F.coalesce(
            F.col("CancelledNum") == 1, F.lit(False)
        )
    )
    .withColumn(
        "Reason_Diverted",
        F.coalesce(
            F.col("DivertedNum") == 1, F.lit(False)
        )
    )
    .withColumn(
        "Reason_InvalidFlightStatus",
        ~F.coalesce(valid_status, F.lit(False))
    )
    .withColumn(
        "Reason_MissingArrivalDelay",
        ~F.coalesce(valid_arrival, F.lit(False))
    )
    .withColumn(
        "Reason_InvalidDelayLabel",
        ~F.coalesce(valid_target, F.lit(False))
    )
)

# Exact combined validation rule used in production.
all_valid = (
    valid_date & valid_airports & valid_time
    & valid_status & valid_arrival & valid_target
)

df = df.withColumn(
    "QualityStatus",
    F.when(all_valid, "ACCEPTED").otherwise("REJECTED")
)

summary = {
    r["QualityStatus"]: r["count"]
    for r in df.groupBy("QualityStatus").count().collect()
}

total = sum(summary.values())
accepted = summary.get("ACCEPTED", 0)
rejected = summary.get("REJECTED", 0)

print("REAL 2025 DATA QUALITY ANALYSIS")
print("Total records:", total)
print("Accepted records:", accepted)
print("Rejected records:", rejected)
print(f"Acceptance rate: {accepted / total:.2%}")

assert total == 7001619
assert accepted == 6879484
assert rejected == 122135

print("PASS: Results match the production ETL")


In [ ]:

# Count each rejection reason.
# A flight can have more than one reason.

reason_columns = {
    "Invalid date or year":
        "Reason_InvalidDate",
    "Missing airline or airport":
        "Reason_MissingAirlineOrAirport",
    "Invalid scheduled departure time":
        "Reason_InvalidDepartureTime",
    "Cancelled flight":
        "Reason_Cancelled",
    "Diverted flight":
        "Reason_Diverted",
    "Invalid cancellation/diversion status":
        "Reason_InvalidFlightStatus",
    "Missing arrival delay":
        "Reason_MissingArrivalDelay",
    "Invalid or inconsistent delay label":
        "Reason_InvalidDelayLabel"
}

rejected_df = df.filter(F.col("QualityStatus") == "REJECTED")

aggregates = [
    F.sum(
        F.when(F.col(column), 1).otherwise(0)
    ).alias(column)
    for column in reason_columns.values()
]

result = rejected_df.agg(*aggregates).first()

reason_data = [
    (name, int(result[column] or 0))
    for name, column in reason_columns.items()
]

reason_df = spark.createDataFrame(
    reason_data,
    ["RejectionReason", "AffectedRecords"]
).withColumn(
    "PercentOfRejected",
    F.round(F.col("AffectedRecords") * 100 / rejected, 2)
).orderBy(F.desc("AffectedRecords"))

print("REJECTION REASONS — REAL 2025 DATA")
display(reason_df)

print("PASS: Rejection-reason analysis completed")
print("Note: One flight may have multiple reasons.")


In [ ]:

# Assign one primary rejection reason using priority order.

primary_reason = (
    F.when(F.col("Reason_InvalidDate"), "Invalid date or year")
    .when(
        F.col("Reason_MissingAirlineOrAirport"),
        "Missing airline or airport"
    )
    .when(
        F.col("Reason_InvalidDepartureTime"),
        "Invalid departure time"
    )
    .when(F.col("Reason_Cancelled"), "Cancelled flight")
    .when(F.col("Reason_Diverted"), "Diverted flight")
    .when(
        F.col("Reason_InvalidFlightStatus"),
        "Invalid flight status"
    )
    .when(
        F.col("Reason_MissingArrivalDelay"),
        "Missing arrival delay"
    )
    .when(
        F.col("Reason_InvalidDelayLabel"),
        "Invalid delay label"
    )
    .otherwise("Other validation failure")
)

primary_df = (
    rejected_df
    .withColumn("PrimaryReason", primary_reason)
    .groupBy("PrimaryReason")
    .count()
    .withColumn(
        "Percentage",
        F.round(F.col("count") * 100 / rejected, 2)
    )
    .orderBy(F.desc("count"))
)

display(primary_df)

primary_total = primary_df.agg(
    F.sum("count").alias("total")
).first()["total"]

assert primary_total == rejected

print("Total rejected records:", primary_total)
print("PASS: Every rejected record has one primary reason")
